# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
# View the dataset metadata (JSON-LD)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List all record sets and their fields by @id (Croissant)

record_sets = dataset.metadata.record_sets
if not record_sets:
    print("No record sets found in the dataset metadata.")
else:
    for record_set in record_sets:
        print(f"\nRecord Set: {record_set['@id']}")
        print(f"  Name: {record_set.get('name','')}")
        print("  Fields:")
        for field in record_set.get('fields', []):
            print(f"    - @id: {field['@id']} | Name: {field.get('name','')} | dataType: {field.get('dataType', '')}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Identify all record set @ids
record_sets = dataset.metadata.record_sets
record_set_ids = [rec["@id"] for rec in record_sets] if record_sets else []
dataframes = {}

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"\nLoaded DataFrame for record set @id: {record_set_id}")
        print("Columns:", df.columns.tolist())
        display(df.head(3))
    else:
        print(f"No records found for record set @id: {record_set_id}")
# You can choose one record set @id to explore further below


## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# Example: pick the first available record set for EDA
if record_set_ids and record_set_ids[0] in dataframes:
    record_set_id = record_set_ids[0]
    df = dataframes[record_set_id]
    print(f"Using record set: {record_set_id}")
    print("Available columns:", df.columns.tolist())

    # Try to find a likely numeric field by data type (or by scanning first record)
    potential_numeric_fields = []
    if dataset.metadata.record_sets:
        recs = dataset.metadata.record_sets
        for rec in recs:
            if rec["@id"] == record_set_id:
                for f in rec.get("fields", []):
                    # Accept integer or float types
                    if f.get('dataType', '').lower() in ['float', 'number', 'integer', 'schema:number', 'schema:integer', 'schema:float']:
                        potential_numeric_fields.append(f["@id"])
    if not potential_numeric_fields:
        # Heuristically pick columns with numeric data
        for c in df.columns:
            if pd.api.types.is_numeric_dtype(df[c]):
                potential_numeric_fields.append(c)

    if potential_numeric_fields:
        numeric_field = potential_numeric_fields[0]
        threshold = df[numeric_field].mean() if df[numeric_field].dtype in ['float64', 'int64'] else 10
        # Filter
        filtered_df = df[df[numeric_field] > threshold].copy()
        print(f"Filtered records with {numeric_field} > {threshold} (using @id):")
        display(filtered_df.head())
        # Normalize
        filtered_df[f"{numeric_field}_normalized"] = (
            filtered_df[numeric_field] - filtered_df[numeric_field].mean()
        ) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered records:")
        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())
        # Group (pick first categorical column that's not the numeric)
        group_field = None
        for c in df.columns:
            if c != numeric_field and df[c].dtype == object:
                group_field = c
                break
        if group_field:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            print(f"Grouped data by {group_field}: (mean {numeric_field})")
            display(grouped_df)
    else:
        print("No numeric field found in this record set.")
else:
    print("No suitable data for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Visualize distribution for the numeric field if analysis was possible earlier
if 'filtered_df' in locals() and not filtered_df.empty and 'numeric_field' in locals():
    plt.figure(figsize=(8,4))
    sns.histplot(filtered_df[numeric_field], kde=True)
    plt.title(f"Distribution of {numeric_field} (filtered, record set: {record_set_id})")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()
    # If a group_field exists, boxplot
    if 'group_field' in locals() and group_field:
        plt.figure(figsize=(10,5))
        sns.boxplot(data=filtered_df, x=group_field, y=numeric_field)
        plt.title(f"Boxplot of {numeric_field} by {group_field}")
        plt.xticks(rotation=30)
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

In this notebook, we loaded a FAIR^2-compliant proteomics dataset using the Croissant standard and the `mlcroissant` library, explored its schema and available record sets, and performed basic exploratory analysis and visualizations on the quantitative data fields. This workflow can be extended to support annotation, feature engineering, or domain-specific mass spectrometry analysis using the provided record set and field `@id` identifiers. For more advanced usage, see the [mlcroissant documentation](https://mlcommons.github.io/croissant/python/).
